<a href="https://colab.research.google.com/github/fernandocesaraviles-tech/ai-bias-audit/blob/main/src/01_load_and_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
01_load_and_explore.py

Paso 1 del proyecto: cargar el dataset Adult Income (Census Income) y hacer
una primera exploración enfocada en detectar señales de sesgo en los DATOS
CRUDOS, antes de entrenar cualquier modelo.

Tarea del dataset: predecir si una persona gana más o menos de 50k USD/año
a partir de variables demográficas y laborales.
"""

import pandas as pd
from sklearn.datasets import fetch_openml

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


def load_data() -> pd.DataFrame:
    """Descarga (o carga desde caché local) el dataset Adult Income."""
    adult = fetch_openml(name="adult", version=2, as_frame=True)
    df = adult.frame.copy()

    if "class" in df.columns and "income" not in df.columns:
        df = df.rename(columns={"class": "income"})

    return df


def resumen_general(df: pd.DataFrame) -> None:
    print("=" * 70)
    print("RESUMEN GENERAL DEL DATASET")
    print("=" * 70)
    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "  (sin nulos)")
    print("\nPrimeras filas:")
    print(df.head())


def distribucion_objetivo(df: pd.DataFrame, target_col: str = "income") -> None:
    print("\n" + "=" * 70)
    print("DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (income)")
    print("=" * 70)
    print(df[target_col].value_counts(normalize=True).round(3))


def tasa_positiva_por_grupo(df: pd.DataFrame, atributo: str, target_col: str = "income") -> pd.Series:
    """
    Calcula la proporción de personas con ingreso '>50K' dentro de cada
    grupo del atributo sensible indicado (sex, race, etc).
    """
    tasa = df.groupby(atributo)[target_col].apply(
        lambda s: s.astype(str).str.contains(">50K").mean()
    )
    return tasa.sort_values(ascending=False)


def indice_impacto_dispar(tasas: pd.Series) -> float:
    """
    Calcula el 'disparate impact ratio': tasa del grupo con peor resultado
    dividido tasa del grupo con mejor resultado. Un valor por debajo de 0.8
    suele tomarse como señal de alerta (regla del 80%, usada en EEOC/EEUU).
    """
    return round(tasas.min() / tasas.max(), 3)


def main():
    df = load_data()

    resumen_general(df)
    distribucion_objetivo(df)

    print("\n" + "=" * 70)
    print("TASA DE INGRESO '>50K' POR SEXO")
    print("=" * 70)
    tasas_sexo = tasa_positiva_por_grupo(df, "sex")
    print(tasas_sexo.round(3))
    print(f"\nÍndice de impacto dispar (sexo): {indice_impacto_dispar(tasas_sexo)}")
    print("(valores < 0.8 son señal de alerta según la 'regla del 80%')")

    print("\n" + "=" * 70)
    print("TASA DE INGRESO '>50K' POR RAZA")
    print("=" * 70)
    tasas_raza = tasa_positiva_por_grupo(df, "race")
    print(tasas_raza.round(3))
    print(f"\nÍndice de impacto dispar (raza): {indice_impacto_dispar(tasas_raza)}")


if __name__ == "__main__":
    main()

RESUMEN GENERAL DEL DATASET
Filas: 48,842 | Columnas: 15

Tipos de datos:
age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
income            category
dtype: object

Valores nulos por columna:
workclass         2799
occupation        2809
native-country     857
dtype: int64

Primeras filas:
   age  workclass  fnlwgt     education  education-num      marital-status         occupation relationship   race  \
0   25    Private  226802          11th              7       Never-married  Machine-op-inspct    Own-child  Black   
1   38    Private   89814       HS-grad              9  Married-civ-spouse    Farming-fishing      Husband  White   
2   28  Local-gov  336951    

/tmp/ipykernel_524/1998204186.py:56: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa = df.groupby(atributo)[target_col].apply(
/tmp/ipykernel_524/1998204186.py:56: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa = df.groupby(atributo)[target_col].apply(


In [3]:
"""
02_baseline_model.py

Paso 2: entrenar un modelo base (regresión logística) para predecir el
ingreso, y comparar el sesgo en las PREDICCIONES del modelo contra el
sesgo que ya vimos en los datos crudos (paso 1).
"""

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)


def load_data() -> pd.DataFrame:
    adult = fetch_openml(name="adult", version=2, as_frame=True)
    df = adult.frame.copy()
    if "class" in df.columns and "income" not in df.columns:
        df = df.rename(columns={"class": "income"})
    df = df.dropna()  # simplificación para el modelo base: descartamos filas con nulos
    return df


def preparar_datos(df: pd.DataFrame):
    y = (df["income"].astype(str).str.contains(">50K")).astype(int)
    X = df.drop(columns=["income"])
    # convertimos columnas categóricas (sex, race, workclass, etc.) a
    # variables numéricas mediante one-hot encoding
    X = pd.get_dummies(X, drop_first=True)
    return X, y


def tasa_positiva_prediccion(df_test: pd.DataFrame, y_pred, atributo: str) -> pd.Series:
    temp = df_test.copy()
    temp["prediccion"] = y_pred
    return temp.groupby(atributo, observed=True)["prediccion"].mean().sort_values(ascending=False)


def indice_impacto_dispar(tasas: pd.Series) -> float:
    return round(tasas.min() / tasas.max(), 3)


def main():
    df = load_data()
    X, y = preparar_datos(df)

    X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
        X, y, df, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_train_scaled, y_train)

    y_pred = modelo.predict(X_test_scaled)

    print("=" * 70)
    print("DESEMPEÑO DEL MODELO BASE")
    print("=" * 70)
    print(f"Precisión (accuracy): {accuracy_score(y_test, y_pred):.3f}")
    print("\nReporte de clasificación:")
    print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"]))

    print("\n" + "=" * 70)
    print("SESGO EN LAS PREDICCIONES DEL MODELO (no en los datos crudos)")
    print("=" * 70)

    tasas_sexo_pred = tasa_positiva_prediccion(df_test, y_pred, "sex")
    print("\nTasa de predicción '>50K' por sexo:")
    print(tasas_sexo_pred.round(3))
    print(f"Índice de impacto dispar (sexo, predicciones): {indice_impacto_dispar(tasas_sexo_pred)}")

    tasas_raza_pred = tasa_positiva_prediccion(df_test, y_pred, "race")
    print("\nTasa de predicción '>50K' por raza:")
    print(tasas_raza_pred.round(3))
    print(f"Índice de impacto dispar (raza, predicciones): {indice_impacto_dispar(tasas_raza_pred)}")

    print("\n" + "=" * 70)
    print("COMPARACIÓN: DATOS CRUDOS (paso 1) vs PREDICCIONES DEL MODELO")
    print("=" * 70)
    print("Datos crudos        -> sexo: 0.36   | raza: 0.435")
    print(f"Predicciones modelo -> sexo: {indice_impacto_dispar(tasas_sexo_pred)}  | raza: {indice_impacto_dispar(tasas_raza_pred)}")


if __name__ == "__main__":
    main()

DESEMPEÑO DEL MODELO BASE
Precisión (accuracy): 0.846

Reporte de clasificación:
              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.90      6803
        >50K       0.73      0.60      0.66      2242

    accuracy                           0.85      9045
   macro avg       0.80      0.76      0.78      9045
weighted avg       0.84      0.85      0.84      9045


SESGO EN LAS PREDICCIONES DEL MODELO (no en los datos crudos)

Tasa de predicción '>50K' por sexo:
sex
Male      0.262
Female    0.078
Name: prediccion, dtype: float64
Índice de impacto dispar (sexo, predicciones): 0.299

Tasa de predicción '>50K' por raza:
race
Asian-Pac-Islander    0.283
White                 0.215
Black                 0.092
Amer-Indian-Eskimo    0.089
Other                 0.062
Name: prediccion, dtype: float64
Índice de impacto dispar (raza, predicciones): 0.221

COMPARACIÓN: DATOS CRUDOS (paso 1) vs PREDICCIONES DEL MODELO
Datos crudos        -> sexo: 0.36   | r

In [4]:
 !pip install fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 4.6 MB/s eta 0:00:00


In [5]:
"""
03_fairness_audit.py

Paso 3: mitigar el sesgo detectado en el modelo base usando Fairlearn,
aplicando post-procesamiento sobre las predicciones para igualar la tasa
de selección entre hombres y mujeres (constraint: demographic_parity).

IMPORTANTE: en Colab, antes de correr este script, ejecutá en una celda
aparte:
    !pip install fairlearn
"""

import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from fairlearn.postprocessing import ThresholdOptimizer

pd.set_option("display.max_columns", None)


def load_data() -> pd.DataFrame:
    adult = fetch_openml(name="adult", version=2, as_frame=True)
    df = adult.frame.copy()
    if "class" in df.columns and "income" not in df.columns:
        df = df.rename(columns={"class": "income"})
    df = df.dropna()
    return df


def preparar_datos(df: pd.DataFrame):
    y = (df["income"].astype(str).str.contains(">50K")).astype(int)
    X = df.drop(columns=["income"])
    X = pd.get_dummies(X, drop_first=True)
    return X, y


def tasa_positiva_prediccion(df_test: pd.DataFrame, y_pred, atributo: str) -> pd.Series:
    temp = df_test.copy()
    temp["prediccion"] = y_pred
    return temp.groupby(atributo, observed=True)["prediccion"].mean().sort_values(ascending=False)


def indice_impacto_dispar(tasas: pd.Series) -> float:
    return round(tasas.min() / tasas.max(), 3)


def main():
    df = load_data()
    X, y = preparar_datos(df)

    X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
        X, y, df, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- Modelo base (igual al paso 2) ---
    modelo_base = LogisticRegression(max_iter=1000)
    modelo_base.fit(X_train_scaled, y_train)
    y_pred_base = modelo_base.predict(X_test_scaled)

    # --- Mitigación con Fairlearn ---
    # Ajustamos el umbral de decisión por separado para cada grupo de sexo,
    # buscando que la proporción de predicciones ">50K" sea similar entre
    # hombres y mujeres (demographic parity), sin re-entrenar el modelo.
    mitigador = ThresholdOptimizer(
        estimator=modelo_base,
        constraints="demographic_parity",
        predict_method="predict_proba",
        prefit=True,
    )
    mitigador.fit(X_train_scaled, y_train, sensitive_features=df_train["sex"])
    y_pred_mitigado = mitigador.predict(X_test_scaled, sensitive_features=df_test["sex"])

    print("=" * 70)
    print("ACCURACY: MODELO BASE vs MODELO MITIGADO")
    print("=" * 70)
    print(f"Accuracy modelo base:     {accuracy_score(y_test, y_pred_base):.3f}")
    print(f"Accuracy modelo mitigado: {accuracy_score(y_test, y_pred_mitigado):.3f}")
    print("(es normal y esperable que la accuracy baje un poco: es el")
    print(" costo de reducir el sesgo — la 'tensión precisión vs equidad')")

    tasas_base = tasa_positiva_prediccion(df_test, y_pred_base, "sex")
    tasas_mitigado = tasa_positiva_prediccion(df_test, y_pred_mitigado, "sex")

    print("\n" + "=" * 70)
    print("ÍNDICE DE IMPACTO DISPAR POR SEXO — ANTES vs DESPUÉS DE MITIGAR")
    print("=" * 70)
    print("Datos crudos (paso 1):     0.36")
    print(f"Modelo base (paso 2):      {indice_impacto_dispar(tasas_base)}")
    print(f"Modelo mitigado (paso 3):  {indice_impacto_dispar(tasas_mitigado)}")

    print("\nTasas de predicción '>50K' por sexo, modelo mitigado:")
    print(tasas_mitigado.round(3))


if __name__ == "__main__":
    main()

ACCURACY: MODELO BASE vs MODELO MITIGADO
Accuracy modelo base:     0.846
Accuracy modelo mitigado: 0.832
(es normal y esperable que la accuracy baje un poco: es el
 costo de reducir el sesgo — la 'tensión precisión vs equidad')

ÍNDICE DE IMPACTO DISPAR POR SEXO — ANTES vs DESPUÉS DE MITIGAR
Datos crudos (paso 1):     0.36
Modelo base (paso 2):      0.299
Modelo mitigado (paso 3):  0.935

Tasas de predicción '>50K' por sexo, modelo mitigado:
sex
Female    0.175
Male      0.163
Name: prediccion, dtype: float64
